# 第17篇｜apply / map / transform：自定义处理，想怎么算就怎么算

> 这是「数据分析从入门到精通」系列的第 17 篇。Pandas 内置函数搞不定的操作，怎么办？这篇教你用 apply / map / transform 自定义处理逻辑，想怎么算就怎么算，彻底解锁 Pandas 的灵活性。

---

嗨，我是小荷～

你有没有遇到过这种情况：数据里有一列"销售额"，你想加一个"销售等级"列——低于 200 标"低"，200~500 标"中"，500 以上标"高"。

用 `groupby` 做不到，用 `loc` 分段赋值写起来也很繁琐……这时候就该 **apply** 出场了。

apply / map / transform 是 Pandas 里的"自定义武器"，把你的逻辑装进函数，然后批量应用到整列/整行/分组里。萧何管粮草的时候，针对不同情况要用不同的处理规则——这种"按规则处理"的思想，就是 apply 的精髓。

---

## 一、map：最简单，用于 Series 逐元素替换

`map` 只能用在 **Series（单列）** 上，适合简单的值替换：


In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'name':  ['张三', '李四', '王五', '赵六'],
    'city':  ['北京', '上海', '广州', '深圳'],
    'score': [85, 72, 91, 63]
})

# 用字典映射：城市 → 区域
city_to_region = {'北京': '华北', '上海': '华东', '广州': '华南', '深圳': '华南'}
df['region'] = df['city'].map(city_to_region)
print(df)


  name city  score region
0   张三   北京     85     华北
1   李四   上海     72     华东
2   王五   广州     91     华南
3   赵六   深圳     63     华南


`map` 还可以传函数：


In [2]:
# 分数乘以1.1作为加权分
df['weighted_score'] = df['score'].map(lambda x: round(x * 1.1, 1))
print(df['weighted_score'])


0     93.5
1     79.2
2    100.1
3     69.3
Name: weighted_score, dtype: float64


> 💡 **map 的局限**：只能处理单列，不能跨列运算。

---

## 二、apply：万能，Series 和 DataFrame 都能用

### 2.1 apply 作用于 Series（单列）

apply 是 Pandas 里最灵活的方法，什么都能干：


In [3]:
# 根据分数打标签
def score_label(score):
    if score >= 90:
        return '优秀'
    elif score >= 75:
        return '良好'
    elif score >= 60:
        return '及格'
    else:
        return '不及格'

df['label'] = df['score'].apply(score_label)
print(df[['name', 'score', 'label']])


  name  score label
0   张三     85    良好
1   李四     72    及格
2   王五     91    优秀
3   赵六     63    及格


当然也可以用 lambda：


In [5]:
df['label'] = df['score'].apply(lambda x: '优秀' if x >= 90 else ('良好' if x >= 75 else '及格'))
df

,name,city,score,region,weighted_score,label
0,张三,北京,85,华北,93.5,良好
1,李四,上海,72,华东,79.2,及格
2,王五,广州,91,华南,100.1,优秀
3,赵六,深圳,63,华南,69.3,及格


---

### 2.2 apply 作用于 DataFrame（整行/整列）

传给 DataFrame 的 `apply` 时，函数会接收整行（axis=1）或整列（axis=0）：


In [6]:
df2 = pd.DataFrame({
    'product': ['A', 'B', 'C', 'D'],
    'price':   [100, 200, 150, 300],
    'cost':    [60,  120, 80,  200]
})

# 计算利润率：(price - cost) / price，跨列运算
df2['profit_rate'] = df2.apply(lambda row: round((row['price'] - row['cost']) / row['price'], 3), axis=1)
print(df2)


  product  price  cost  profit_rate
0       A    100    60        0.400
1       B    200   120        0.400
2       C    150    80        0.467
3       D    300   200        0.333


`axis=1` 表示对每一**行**应用函数（函数接收一行数据），`axis=0` 表示对每一**列**。

---

### 2.3 apply 返回多列

有时候一个 apply 可以同时生成多列：


In [7]:
def analyze_score(score):
    label = '优秀' if score >= 90 else '良好' if score >= 75 else '及格' if score >= 60 else '不及格'
    bonus = score * 0.1 if score >= 90 else 0
    return pd.Series({'label': label, 'bonus': bonus})

result = df['score'].apply(analyze_score)
print(result)
df = pd.concat([df, result], axis=1)
print(df)


  label  bonus
0    良好    0.0
1    及格    0.0
2    优秀    9.1
3    及格    0.0
  name city  score region  weighted_score label label  bonus
0   张三   北京     85     华北            93.5    良好    良好    0.0
1   李四   上海     72     华东            79.2    及格    及格    0.0
2   王五   广州     91     华南           100.1    优秀    优秀    9.1
3   赵六   深圳     63     华南            69.3    及格    及格    0.0


---

## 三、applymap（Pandas 2.1+ 改名为 map）：对 DataFrame 每个格子处理

`applymap`（Pandas 2.1+ 改名为 `map`）可以对 DataFrame 的**每一个格子**应用同一个函数，比如把所有数字格式化成带单位的字符串：


In [8]:
df_nums = pd.DataFrame({
    'math':    [78, 92, 85, 63],
    'english': [88, 76, 90, 72],
    'science': [70, 88, 82, 95]
})

# 所有分数都格式化成带单位的字符串
formatted = df_nums.map(lambda x: f"{x}分")
print(formatted)


  math english science
0  78分     88分     70分
1  92分     76分     88分
2  85分     90分     82分
3  63分     72分     95分


---

## 四、transform：分组内计算，保留原始索引

`transform` 最常配合 `groupby` 使用，特点是**结果和原 DataFrame 等长**，不会把分组压缩掉：


In [9]:
sales_df = pd.DataFrame({
    'city':  ['北京', '北京', '上海', '上海', '广州', '广州'],
    'sales': [200, 300, 150, 250, 400, 350]
})

# 计算每个城市的总销售额，但保留每行
sales_df['city_total'] = sales_df.groupby('city')['sales'].transform('sum')

# 计算每行销售额占该城市的比例
sales_df['city_share'] = (sales_df['sales'] / sales_df['city_total']).round(3)

print(sales_df)


  city  sales  city_total  city_share
0   北京    200         500       0.400
1   北京    300         500       0.600
2   上海    150         400       0.375
3   上海    250         400       0.625
4   广州    400         750       0.533
5   广州    350         750       0.467


> 💡 **transform vs groupby().agg()**：`agg` 是聚合（每组出一行），`transform` 是变换（每行都保留，填入分组计算值）。

---

## 五、三者对比

| 方法 | 作用对象 | 用途 | 返回形状 |
|------|---------|------|---------|
| `map` | Series | 简单值映射/替换 | 等长 Series |
| `apply` | Series / DataFrame | 自定义函数，可跨列 | 灵活（标量/Series/DataFrame） |
| `transform` | 配合 groupby | 分组计算，保留原长 | 等长 Series/DataFrame |

---

## 六、🔧 综合实战：批量打标签 + 复杂列计算

学了一堆理论，来个完整的实战练练手——把前面学的知识点串起来：


In [11]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200

df = pd.DataFrame({
    'order_id':   range(1001, 1001+n),
    'user_id':    np.random.choice(range(101, 121), n),
    'amount':     np.random.randint(50, 2000, n),
    'quantity':   np.random.randint(1, 10, n),
    'discount':   np.random.choice([0, 0.05, 0.1, 0.2], n),
    'city':       np.random.choice(['北京', '上海', '广州', '深圳'], n),
    'category':   np.random.choice(['数码', '服装', '食品'], n)
})

# ── 任务1：计算实际支付金额 ──
df['actual_amount'] = df.apply(
    lambda row: round(row['amount'] * (1 - row['discount']), 2), axis=1
)

# ── 任务2：给订单打金额等级标签 ──
def amount_tier(x):
    if x >= 1000: return '大单'
    elif x >= 500: return '中单'
    else: return '小单'

df['order_tier'] = df['actual_amount'].apply(amount_tier)

# ── 任务3：计算每个用户的历史累计消费 ──
df['user_cumsum'] = df.groupby('user_id')['actual_amount'].transform('cumsum').round(2)

# ── 任务4：给用户打价值标签（基于该用户总消费） ──
user_total = df.groupby('user_id')['actual_amount'].transform('sum')
df['user_value'] = user_total.apply(lambda x: '高价值' if x >= 5000 else '中价值' if x >= 2000 else '普通用户')

# ── 查看结果 ──
print(df[['order_id', 'amount', 'discount', 'actual_amount', 'order_tier', 'user_value']].head(10))

# ── 统计各等级订单数 ──
print("\n订单等级分布：")
print(df['order_tier'].value_counts())

print("\n用户价值分布：")
print(df['user_value'].value_counts())


   order_id  amount  discount  actual_amount order_tier user_value
0      1001    1419      0.00        1419.00         大单        高价值
1      1002    1584      0.05        1504.80         大单        高价值
2      1003     196      0.00         196.00         小单        高价值
3      1004    1221      0.05        1159.95         大单        高价值
4      1005     913      0.05         867.35         中单        高价值
5      1006    1784      0.20        1427.20         大单        高价值
6      1007    1893      0.00        1893.00         大单        高价值
7      1008     538      0.20         430.40         小单        高价值
8      1009     978      0.00         978.00         中单        高价值
9      1010    1713      0.00        1713.00         大单        高价值

订单等级分布：
order_tier
大单    89
中单    60
小单    51
Name: count, dtype: int64

用户价值分布：
user_value
高价值    200
Name: count, dtype: int64


---

## 七、⚡ 性能提示

`apply` 灵活但速度相对较慢，因为它是逐行/逐列的 Python 循环。能用向量化操作的，优先用向量化：


In [ ]:
# ❌ 慢：用 apply 做简单四则运算
df['result'] = df['price'].apply(lambda x: x * 1.13)

# ✅ 快：直接向量化
df['result'] = df['price'] * 1.13


> 💡 **记住**：apply 适合"复杂逻辑"，简单运算别用它。

---

## 八、📝 小结

| 方法 | 最适合的场景 |
|------|------------|
| `map` | 单列的值替换、字典映射 |
| `apply`（Series） | 单列的复杂判断/计算 |
| `apply`（DataFrame, axis=1） | 跨多列的行级运算 |
| `transform` | groupby 后计算分组值、保留原始行数 |

---

## 九、🏋️ 课后练习

1. 给一列收入数据（0~50000）打区间标签：`<5000 = '低收入'`, `5000~15000 = '中等收入'`, `>15000 = '高收入'`。
2. 用 `apply(axis=1)` 计算每行的"折扣后利润率"（需要同时用到 `price` 和 `cost` 两列）。
3. 用 `transform` 计算每个品类的中位数消费，并新增一列标出每条订单是"高于中位数"还是"低于中位数"。

本篇完整代码包括练习题解答都已经上传至 GitHub 仓库，欢迎 Clone。

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')

# seaborn 主题会覆盖字体设置，重新指定中文字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ========== 构造示例数据 ==========
np.random.seed(2024)
n = 200
categories = ['数码', '服饰', '食品', '家居', '美妆']
df = pd.DataFrame({
    '订单ID': range(1001, 1001 + n),
    '品类': np.random.choice(categories, n),
    '收入': np.random.uniform(0, 50000, n).round(2),
    'price': np.random.uniform(50, 500, n).round(2),
    'cost': np.random.uniform(20, 300, n).round(2),
    '折扣': np.random.uniform(0.5, 1.0, n).round(2)
})
print("数据预览:")
print(df.head())

# ========== 任务1：给收入数据打区间标签 ==========
print("\n" + "=" * 50)
print("任务1：收入区间标签")
print("=" * 50)
def income_label(x):
    if x < 5000:
        return '低收入'
    elif x <= 15000:
        return '中等收入'
    else:
        return '高收入'

df['收入等级'] = df['收入'].apply(income_label)
print(df['收入等级'].value_counts())
print(df[['收入', '收入等级']].head(10))

# ========== 任务2：用 apply(axis=1) 计算每行的折扣后利润率 ==========
print("\n" + "=" * 50)
print("任务2：计算折扣后利润率")
print("=" * 50)
def calc_profit_margin(row):
    discounted_price = row['price'] * row['折扣']
    profit = discounted_price - row['cost']
    margin = profit / discounted_price * 100 if discounted_price != 0 else 0
    return round(margin, 2)

df['折扣后利润率(%)'] = df.apply(calc_profit_margin, axis=1)
print(df[['price', 'cost', '折扣', '折扣后利润率(%)']].head(10))
print(f"\n平均折扣后利润率: {df['折扣后利润率(%)'].mean():.2f}%")

# ========== 任务3：用 transform 计算每个品类的中位数消费，标注高低 ==========
print("\n" + "=" * 50)
print("任务3：每条订单是否高于品类中位数")
print("=" * 50)
median_by_cat = df.groupby('品类')['收入'].transform('median')
df['品类中位数'] = median_by_cat.round(2)
df['相对中位数'] = df.apply(lambda row: '高于中位数' if row['收入'] > row['品类中位数'] else '低于中位数', axis=1)

print("各品类中位数:")
print(df.groupby('品类')['收入'].median().round(2))
print(f"\n各品类高低分布:")
print(pd.crosstab(df['品类'], df['相对中位数']))
print(f"\n详细数据:")
print(df[['品类', '收入', '品类中位数', '相对中位数']].head(10))

数据预览:
   订单ID  品类        收入   price    cost    折扣
0  1001  数码  12177.47  414.70  178.94  0.62
1  1002  食品  41909.28  144.59  153.31  0.86
2  1003  数码   3531.37  266.02  112.68  0.62
3  1004  数码  25842.86  411.55  209.99  0.88
4  1005  家居  16542.25  276.75  294.94  1.00

任务1：收入区间标签
收入等级
高收入     145
中等收入     40
低收入      15
Name: count, dtype: int64
         收入  收入等级
0  12177.47  中等收入
1  41909.28   高收入
2   3531.37   低收入
3  25842.86   高收入
4  16542.25   高收入
5   1201.63   低收入
6   7448.73  中等收入
7  49144.76   高收入
8  19557.33   高收入
9   4875.87   低收入

任务2：计算折扣后利润率
    price    cost    折扣  折扣后利润率(%)
0  414.70  178.94  0.62      30.40
1  144.59  153.31  0.86     -23.29
2  266.02  112.68  0.62      31.68
3  411.55  209.99  0.88      42.02
4  276.75  294.94  1.00      -6.57
5  457.59  116.40  0.89      71.42
6  448.47  237.40  0.74      28.47
7  298.95  277.26  0.67     -38.42
8  137.83  199.46  0.69    -109.73
9  155.81   33.43  0.61      64.83

平均折扣后利润率: -4.94%

任务3：每条订单是否高于品类中位数
各品类中位数:
品类
家居    

---

## 下期预告

> **第 18 篇：阶段大实战 — 电商用户行为分析**
>
> Pandas 的 18 篇学习走到这里，是时候来一场综合大实战了！从数据读取、清洗、合并、聚合到报告输出，完整走一遍真实项目流程——这个项目可以直接写进你的作品集。

---

*跟着小荷，数据分析路上不迷路～*
*（每一个 apply 背后，都是萧何式的"因地制宜"～哈哈）*